<a href="https://colab.research.google.com/github/askSayyam/CrisisLens/blob/main/Embedding%20%26%20FAISS%20index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#PHASE 2 - loading LABSE model , Endcoding and BUILding FAISS Index

In [ ]:
#MOUNTING THE DRIVE
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os

DRIVE = '/content/drive/MyDrive/CrisisLens/'
print("Drive mounted ✓")

In [ ]:
#loading the files and labse models path from drive
import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer

# ── Load all 5 files ──
unified    = pd.read_csv(DRIVE + 'unified_corpus.csv')
queries_df = pd.read_csv(DRIVE + 'queries.csv')
qrels_df   = pd.read_csv(DRIVE + 'qrels.csv')
posts_df   = pd.read_csv(DRIVE + 'posts_clean.csv')
pairs_df   = pd.read_csv(DRIVE + 'pairs_dev_crosslingual.csv')

print(f"unified_corpus      : {len(unified):,} rows")
print(f"queries             : {len(queries_df):,} rows")
print(f"qrels               : {len(qrels_df):,} rows")
print(f"posts_clean         : {len(posts_df):,} rows")
print(f"pairs_crosslingual  : {len(pairs_df):,} rows")
print("All 5 files loaded ✓")

# ── Load LaBSE ──
LABSE_PATH = DRIVE + 'labse_model/'

if os.path.exists(LABSE_PATH):
    print("\nLoading LaBSE from Drive cache...")
    model = SentenceTransformer(LABSE_PATH)
else:
    print("\nDownloading LaBSE for the first time...")
    model = SentenceTransformer('sentence-transformers/LaBSE')
    model.save(LABSE_PATH)
    print("LaBSE saved to Drive ✓")

print(f"LaBSE ready ✓ Dim: {model.get_sentence_embedding_dimension()}")
print("\nAll loaded. Phase 2 ready ✓")

In [ ]:
#sanity check
test_queries = [
    ("eng", "flood evacuation route"),
    ("urd", "سیلاب سے بچاؤ کا راستہ"),
    ("ara", "طريق الإخلاء من الفيضانات"),
    ("fra", "itinéraire évacuation inondations"),
]

test_passages = [
    "Emergency flood evacuation routes and shelter locations",
    "Earthquake damage assessment and rescue operations",
]

q_vecs = model.encode([q[1] for q in test_queries], normalize_embeddings=True)
p_vecs = model.encode(test_passages, normalize_embeddings=True)
scores = q_vecs @ p_vecs.T

print(f"{'Language':<6} {'Query':<35} {'Flood score':>12} {'Quake score':>12}")
print("-" * 70)
for i, (lang, q) in enumerate(test_queries):
    print(f"{lang:<6} {q[:35]:<35} {scores[i,0]:>12.3f} {scores[i,1]:>12.3f}")

print("\nAll flood queries must score HIGHER on flood passage ✓")

In [ ]:
import numpy as np
from tqdm import tqdm
import os

texts = unified['text'].fillna('').tolist()
uids  = unified['uid'].tolist()
N     = len(texts)
BATCH = 256
DIM   = 768

print(f"Total passages : {N:,}")
print(f"Batch size     : {BATCH}")
print(f"Drive path     : {DRIVE}")

# ── Check Drive is writable before starting ──
test_path = DRIVE + 'write_test.npy'
try:
    np.save(test_path, np.array([1,2,3]))
    os.remove(test_path)
    print("Drive write test   : ✓ writable")
except Exception as e:
    print(f"Drive write FAILED : {e}")
    print("Fix Drive connection before encoding!")
    raise

# ── Check GPU ──
import torch
if torch.cuda.is_available():
    print(f"GPU                : ✓ {torch.cuda.get_device_name(0)}")
else:
    print("GPU NOT FOUND — switch to T4 before running!")
    raise RuntimeError("No GPU")

# ── Check if already partially done ──
checkpoint_files = sorted([
    f for f in os.listdir(DRIVE)
    if f.startswith('embs_checkpoint_')
])
if checkpoint_files:
    print(f"Checkpoints found  : {checkpoint_files}")
    last = checkpoint_files[-1]
    start_from = int(last.replace('embs_checkpoint_','').replace('.npy',''))
    all_embs = np.zeros((N, DIM), dtype='float32')
    partial  = np.load(DRIVE + last)
    all_embs[:partial.shape[0]] = partial
    print(f"Resuming from      : {start_from:,}")
else:
    start_from = 0
    all_embs   = np.zeros((N, DIM), dtype='float32')
    print(f"Starting fresh from: 0")

print("\nStarting encoding...\n")

# ── Encode with checkpoint saving ──
try:
    for start in tqdm(range(start_from, N, BATCH)):
        batch = texts[start : start + BATCH]

        embs = model.encode(
            batch,
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=BATCH,
            convert_to_numpy=True
        )
        all_embs[start : start + len(batch)] = embs

        # Save checkpoint every 50k rows
        if start > 0 and (start // BATCH) % 195 == 0:
            ckpt_path = DRIVE + f'embs_checkpoint_{start}.npy'
            np.save(ckpt_path, all_embs[:start])
            tqdm.write(f"Checkpoint saved at {start:,}")

except Exception as e:
    # Save emergency checkpoint if something crashes
    emergency_path = DRIVE + f'embs_emergency_{start}.npy'
    np.save(emergency_path, all_embs[:start])
    print(f"\nCrash at row {start:,}")
    print(f"Emergency checkpoint saved → {emergency_path}")
    raise

# ── Save final outputs ──
np.save(DRIVE + 'corpus_embeddings.npy', all_embs)
np.save(DRIVE + 'corpus_uids.npy', np.array(uids))

# ── Verify saved correctly ──
loaded_check = np.load(DRIVE + 'corpus_embeddings.npy')
assert loaded_check.shape == (N, DIM), f"Shape mismatch: {loaded_check.shape}"
assert len(np.load(DRIVE + 'corpus_uids.npy', allow_pickle=True)) == N

# ── Delete checkpoints (no longer needed) ──
for f in os.listdir(DRIVE):
    if f.startswith('embs_checkpoint_') or f.startswith('embs_emergency_'):
        os.remove(DRIVE + f)
        print(f"Deleted checkpoint: {f}")

print(f"\n{'='*45}")
print(f"Encoding complete ✓")
print(f"Shape : {loaded_check.shape}")
print(f"UIDs  : {N:,}")
print(f"Saved to Drive permanently ✓")
print(f"{'='*45}")

In [ ]:
#FAISS index
!pip install faiss-cpu
import faiss
import numpy as np

# ── Reload embeddings fresh from Drive ──
print("Loading embeddings from Drive...")
embs = np.load(DRIVE + 'corpus_embeddings.npy').astype('float32')
uids = np.load(DRIVE + 'corpus_uids.npy', allow_pickle=True)
print(f"Embeddings shape : {embs.shape}")
print(f"UIDs count       : {len(uids):,}")

# ── Verify shape before building ──
assert embs.shape == (393424, 768), f"Wrong shape: {embs.shape}"
print("Shape verified ✓")

# ── Build FAISS index ──
print("\nBuilding FAISS index...")
index = faiss.IndexFlatIP(768)
index.add(embs)
print(f"Vectors added : {index.ntotal:,}")

# ── Save to Drive ──
faiss.write_index(index, DRIVE + 'corpus.faiss')
print(f"corpus.faiss saved to Drive ✓")

# ── Verify saved correctly ──
index_check = faiss.read_index(DRIVE + 'corpus.faiss')
assert index_check.ntotal == 393424, f"Mismatch: {index_check.ntotal}"
print(f"Verified ✓ {index_check.ntotal:,} vectors in index")

# ── Quick test ──
test_query = "flood evacuation emergency shelter"
q_vec = model.encode([test_query], normalize_embeddings=True).astype('float32')
D, I  = index.search(q_vec, k=3)

print(f"\nTop 3 results for '{test_query}':")
for rank, (score, idx) in enumerate(zip(D[0], I[0]), 1):
    print(f"  {rank}. [{uids[idx]}] score={score:.3f}")
    print(f"     {unified.iloc[idx]['text'][:80]}...")